# 호출 한도와 비용 상한

호출 전에 예상 비용을 예약해 한도 밖 요청을 모델로 보내지 않습니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

교육용 예상 비용을 호출당 0.002달러로 둡니다. 이 값은 실제 가격표가 아닙니다.

In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 기본 실행

예산은 호출 수와 금액을 각각 검사합니다.

In [4]:
from dataclasses import dataclass
import math

@dataclass
class CallBudget:
    max_calls: int
    max_cost_usd: float
    calls: int = 0
    cost_usd: float = 0.0

    def reserve(self, upper_bound_usd: float) -> None:
        valid = math.isfinite(upper_bound_usd) and upper_bound_usd >= 0
        if not valid or self.calls + 1 > self.max_calls:
            raise ValueError("호출 한도를 초과합니다.")
        if self.cost_usd + upper_bound_usd > self.max_cost_usd:
            raise ValueError("계획 비용 상한을 초과합니다.")
        self.calls += 1
        self.cost_usd += upper_bound_usd

budget = CallBudget(max_calls=2, max_cost_usd=0.005)
estimated = 0.002
budget.reserve(estimated)
answer = model.invoke("LLM 애플리케이션의 가드레일을 소프트웨어 관점에서 한 문장으로 정의하세요. '보장'이라는 표현은 쓰지 마세요.")
print(answer.text)
print("예약 상태:", budget.calls, f"${budget.cost_usd:.3f}")

LLM 애플리케이션의 가드레일은 모델의 입력·출력·행동을 소프트웨어적으로 제어해, 정책에 맞지 않거나 위험한 결과가 나오지 않도록 감시·필터링·완화하는 보호 계층입니다.
예약 상태: 1 $0.002


## 사용량 관찰

실제 반환된 사용량을 따로 읽어 예상값과 구분합니다.

In [5]:
usage = answer.usage_metadata or {}
print("입력 토큰:", usage.get("input_tokens"))
print("출력 토큰:", usage.get("output_tokens"))
print("예상 비용은 수업용 상한 계산값")

입력 토큰: 44
출력 토큰: 63
예상 비용은 수업용 상한 계산값


## 실패 재현

호출 수가 남아 있어도 금액 상한을 넘으면 예약을 거부합니다.

In [6]:
# 예약이 실패하면 model.invoke()를 실행하지 않습니다.
try:
    budget.reserve(0.004)
except ValueError as error:
    print(type(error).__name__, error)

ValueError 계획 비용 상한을 초과합니다.


## 수정

요청을 축소해 남은 범위 안에서 예약합니다.

In [7]:
budget.reserve(0.001)
print("최종 상태:", budget.calls, f"${budget.cost_usd:.3f}")
assert budget.calls == 2 and budget.cost_usd <= budget.max_cost_usd

최종 상태: 2 $0.003
